# Amharic Spell Checker: Evaluation Metrics (Presentation Notebook)

This notebook demonstrates key NLP evaluation metrics that fit this project. Each metric is explained in plain language and computed on the provided corpus/dictionary or on a **synthetic evaluation set** when gold labels are not available.

Metrics covered:
- OOV rate and dictionary coverage
- Perplexity (language model quality)

In [1]:
# Setup: imports and local package path
import math
import random
import sys
from pathlib import Path

# Make src/ importable
ROOT = Path.cwd()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from amharic_spell.preprocessing.normalizer import AmharicNormalizer
from amharic_spell.preprocessing.tokenizer import AmharicTokenizer
from amharic_spell.core.dictionary import Dictionary
from amharic_spell.metrics.edit_distance import calculate_edit_distance
from amharic_spell.models.ngram import InterpolatedLanguageModel
from amharic_spell.core.corrector import SpellCorrector

## Load and preprocess data
We use the corpus for language modeling and the dictionary for coverage/OOV statistics. The tokenizer and normalizer are from the codebase.

**Note:** Normalization can reduce spelling variants; it impacts OOV rate and correction accuracy.

In [2]:
# Paths
corpus_path = ROOT / 'data' / 'amharic_corpus.txt'
dictionary_path = ROOT / 'data' / 'amharic_dictionary.txt'

normalizer = AmharicNormalizer()
tokenizer = AmharicTokenizer()
dictionary = Dictionary(dictionary_path)

# Load raw corpus
raw_text = corpus_path.read_text(encoding='utf-8')

# Normalize and tokenize
normalized_text = normalizer.normalize(raw_text)
sentences = tokenizer.tokenize_sentence(normalized_text)
tokenized_sentences = [tokenizer.tokenize(s) for s in sentences]
tokens = [t for sent in tokenized_sentences for t in sent]

print('Sentences:', len(tokenized_sentences))
print('Tokens:', len(tokens))
print('Dictionary size:', len(dictionary))

Sentences: 424347
Tokens: 6300975
Dictionary size: 95290


## OOV Rate and Dictionary Coverage
**OOV (Out‑Of‑Vocabulary) rate** measures how many tokens are not in the dictionary. Lower OOV indicates better coverage. We report OOV before and after normalization.

- **OOV Rate** = $rac{	ext{tokens not in dictionary}}{	ext{all tokens}}$
- **Coverage** = $1 - 	ext{OOV Rate}$

In [3]:
def compute_oov_rate(tokens_list, dictionary_obj):
    oov = [t for t in tokens_list if t not in dictionary_obj]
    return len(oov) / max(1, len(tokens_list))

# OOV on normalized tokens (already normalized above)
oov_rate_norm = compute_oov_rate(tokens, dictionary)

# OOV on raw tokens (without normalization)
raw_sentences = tokenizer.tokenize_sentence(raw_text)
raw_tokens = [t for s in raw_sentences for t in tokenizer.tokenize(s)]
oov_rate_raw = compute_oov_rate(raw_tokens, dictionary)

print(f'OOV Rate (raw): {oov_rate_raw:.3f} | Coverage: {1-oov_rate_raw:.3f}')
print(f'OOV Rate (normalized): {oov_rate_norm:.3f} | Coverage: {1-oov_rate_norm:.3f}')

OOV Rate (raw): 0.216 | Coverage: 0.784
OOV Rate (normalized): 0.216 | Coverage: 0.784


## Perplexity (Language Model Quality)
**Perplexity** measures how well a language model predicts held‑out text. Lower perplexity means better predictive power.

We compute perplexity on a held‑out subset of the corpus.

In [7]:
def perplexity(model: InterpolatedLanguageModel, tokenized_test):
    log_prob_sum = 0.0
    token_count = 0
    eps = 1e-12

    for sent in tokenized_test:
        context = []
        for w in sent:
            p = model.score(w, context)
            log_prob_sum += math.log(max(p, eps))
            token_count += 1
            context = (context + [w])[-2:]

    return math.exp(-log_prob_sum / max(1, token_count))

# Train/test split
split = int(0.9 * len(tokenized_sentences))
train_sents = tokenized_sentences[:split]
test_sents = tokenized_sentences[split:]

lm_full = InterpolatedLanguageModel()
lm_full.train(train_sents)

pp = perplexity(lm_full, test_sents[:1000])  # limit for speed
print('Perplexity (held‑out):', round(pp, 2))

Perplexity (held‑out): 1973.72
